# 02 - Reversible residual streams: which variant works?

All variants use the *same* blocks (1 attention + 1 MLP per layer, identical parameter count) and only
change how block outputs are combined. In reversible mode the forward pass keeps **no per-layer
activations**; the backward pass reconstructs each layer's input from its output and recomputes it.

| variant | update | inverse |
|---|---|---|
| `coupling` (RevNet / Reformer, "symplectic Euler" on a split state) | y1 = x1 + Attn(x2); y2 = x2 + MLP(y1) | exact: x2 = y2 - MLP(y1); x1 = y1 - Attn(x2) |
| `midpoint` (explicit midpoint / leapfrog) | x_{n+1} = x_{n-1} + 2h f(x_n) | exact: x_{n-1} = x_{n+1} - 2h f(x_n) |
| `momentum` (Momentum ResNet, gamma=0.9) | v' = g v + (1-g) f(x); x' = x + v' | exact, but divides by g |
| `euler` (plain residual) | x' = x + f(x) | **approximate**: fixed-point x <- y - f(x), 10 iters |

Plan: (1) check reconstruction and gradient error vs. plain autograd, (2) 2M-token screening run of
each variant at B=32, (3) train the best one for the full 50M tokens at B=32 (same as the baseline).

In [1]:
# --- setup: works on Google Colab (CUDA) and locally (Apple MPS / CUDA / CPU) ---
import os, sys
# MPS: make the allocator raise a clean OOM at the GPU's recommended working-set size
# instead of spilling into swap (must be set before torch is imported).
os.environ.setdefault("PYTORCH_MPS_HIGH_WATERMARK_RATIO", "1.0")
os.environ.setdefault("PYTORCH_MPS_LOW_WATERMARK_RATIO", "0.9")
if "google.colab" in sys.modules and not os.path.exists("revllm.py"):
    !git clone -q -b session13 https://github.com/chapanduk/ERAV5.git /content/ERAV5
    %cd /content/ERAV5/session13
    !pip -q install tiktoken pyarrow
if not os.path.exists("data/train.bin"):
    !python prepare_data.py
import json, torch, revllm
from revllm import ModelConfig, TrainConfig, train, pick_device
DEVICE = pick_device()
print("torch", torch.__version__, "| device", DEVICE,
      "|", torch.cuda.get_device_name() if DEVICE == "cuda" else "")
TRAIN_BIN, VAL_BIN = "data/train.bin", "data/val.bin"
GiB = 2**30

torch 2.15.0 | device mps | 


## 1. Numerical checks (reconstruction + gradient error)

In [2]:
torch.manual_seed(0)
data = revllm.TokenData(TRAIN_BIN, 512)
x, y = data.batch(0, 4, DEVICE)
checks = {}
for arch in ["coupling", "midpoint", "momentum", "euler"]:
    m = revllm.make_model(ModelConfig(arch=arch), DEVICE)
    checks[arch] = {"recon_err_init": revllm.reconstruction_error(m, x),
                    "grad_err_init": revllm.gradient_error(m, x, y)}
    # stress test: weights scaled up 4x (a proxy for a trained network with larger residual updates)
    with torch.no_grad():
        for n, p in m.named_parameters():
            if p.dim() >= 2 and "blocks" in n: p.mul_(4.0)
    checks[arch]["recon_err_w4x"] = revllm.reconstruction_error(m, x)
    checks[arch]["grad_err_w4x"] = revllm.gradient_error(m, x, y)
    print(arch, {k: f"{v:.2e}" for k, v in checks[arch].items()})
    del m
os.makedirs("results", exist_ok=True)
json.dump(checks, open("results/02_numerical_checks.json", "w"), indent=1)

coupling {'recon_err_init': '6.61e-06', 'grad_err_init': '1.89e-06', 'recon_err_w4x': '9.10e-03', 'grad_err_w4x': '2.69e-04'}


midpoint {'recon_err_init': '6.75e-06', 'grad_err_init': '1.89e-06', 'recon_err_w4x': '1.15e-02', 'grad_err_w4x': '1.54e-04'}


momentum {'recon_err_init': '3.54e-06', 'grad_err_init': '1.66e-06', 'recon_err_w4x': '1.64e-04', 'grad_err_w4x': '2.52e-04'}


euler {'recon_err_init': '1.44e+00', 'grad_err_init': '8.36e-01', 'recon_err_w4x': '3.17e+01', 'grad_err_w4x': '4.97e-01'}


## 2. Screening: 2M tokens per variant (B=32)

In [3]:
SCREEN_TOKENS = 2_000_000
screen = {}
for arch in ["standard", "coupling", "midpoint", "momentum", "euler"]:
    r, m = train(ModelConfig(arch=arch),
                 TrainConfig(name=f"02_screen_{arch}", batch_size=32, total_tokens=SCREEN_TOKENS,
                             eval_every_tokens=SCREEN_TOKENS, log_every=50),
                 TRAIN_BIN, VAL_BIN, DEVICE)
    if arch != "standard":  # reconstruction error of the *trained* weights
        r["recon_err_trained"] = revllm.reconstruction_error(m, x)
    screen[arch] = {k: r.get(k) for k in ["final_train_loss", "final_val_loss", "tokens_per_sec",
                                          "peak_mem_bytes", "recon_err_trained"]}
    del m
for a, s in screen.items():
    print(f"{a:9s} val {s['final_val_loss']:.4f}  {s['tokens_per_sec']/1e3:6.1f}k tok/s  "
          f"peak {s['peak_mem_bytes']/GiB:5.2f} GiB  recon_err {s['recon_err_trained']}")
json.dump(screen, open("results/02_screen_summary.json", "w"), indent=1)

[02_screen_standard] arch=standard params=40.77M (non-emb 21.25M) device=mps B=32 T=512 tokens/step=16384 steps=122


[02_screen_standard] step     1/122 tok   0.02M loss 10.9247 gnorm 8.00   22.9k tok/s peak 7.00GiB


[02_screen_standard] step     2/122 tok   0.03M loss 10.5525 gnorm 6.68   22.9k tok/s peak 7.31GiB


[02_screen_standard] step     3/122 tok   0.05M loss 10.1167 gnorm 1.78   22.0k tok/s peak 7.31GiB


[02_screen_standard] step    50/122 tok   0.82M loss 7.3367 gnorm 0.43   17.0k tok/s peak 7.31GiB


[02_screen_standard] step   100/122 tok   1.64M loss 7.1421 gnorm 0.33   16.8k tok/s peak 7.31GiB


... (36 outputs left out)

[02_screen_euler] step   122/122 tok   2.00M loss 7.5029 gnorm 0.26    5.8k tok/s peak 0.71GiB


[02_screen_euler]   val_loss 7.6188


[02_screen_euler] DONE final_train_loss=8.6732 final_val_loss=7.6188 tok/s=6115 peak=0.71GiB time=5.4min


standard  val 7.1365    16.8k tok/s  peak  7.31 GiB  recon_err None
coupling  val 7.1212    14.2k tok/s  peak  0.78 GiB  recon_err 0.009519902989268303
midpoint  val 7.1142    13.7k tok/s  peak  0.80 GiB  recon_err 0.015462095849215984
momentum  val 7.1092    13.2k tok/s  peak  0.80 GiB  recon_err 0.00012646950199268758
euler     val 7.6188     6.1k tok/s  peak  0.71 GiB  recon_err 153.06240844726562


In [4]:
import math
ok = {a: s for a, s in screen.items() if a != "standard" and math.isfinite(s["final_val_loss"])
      and (s["recon_err_trained"] or 1) < 1e-3}
BEST = min(ok, key=lambda a: ok[a]["final_val_loss"])
print("best reversible variant:", BEST)
json.dump({"best": BEST}, open("results/02_best_variant.json", "w"))

best reversible variant: momentum


## 3. Full 50M-token run with the best reversible variant, B=32 (same as baseline)

In [5]:
res, model = train(ModelConfig(arch=BEST),
                   TrainConfig(name=f"02_reversible_{BEST}_B32", batch_size=32),
                   TRAIN_BIN, VAL_BIN, DEVICE)
print("reconstruction error after training:", revllm.reconstruction_error(model, x))
print("gradient error after training:", revllm.gradient_error(model, x, y))

[02_reversible_momentum_B32] arch=momentum params=40.77M (non-emb 21.25M) device=mps B=32 T=512 tokens/step=16384 steps=3051


[02_reversible_momentum_B32] step     1/3051 tok   0.02M loss 10.9044 gnorm 3.42   17.4k tok/s peak 0.49GiB


[02_reversible_momentum_B32] step     2/3051 tok   0.03M loss 10.8010 gnorm 3.50   12.7k tok/s peak 0.80GiB


[02_reversible_momentum_B32] step     3/3051 tok   0.05M loss 10.6046 gnorm 3.15   11.9k tok/s peak 0.80GiB


[02_reversible_momentum_B32] step    50/3051 tok   0.82M loss 7.1761 gnorm 0.42   12.6k tok/s peak 0.80GiB


[02_reversible_momentum_B32] step   100/3051 tok   1.64M loss 6.7252 gnorm 0.39   13.8k tok/s peak 0.80GiB


... (69 outputs left out)

[02_reversible_momentum_B32]   val_loss 4.7126


[02_reversible_momentum_B32] DONE final_train_loss=4.7357 final_val_loss=4.7126 tok/s=13857 peak=0.80GiB time=60.1min


reconstruction error after training: 0.0001077751730917953


gradient error after training: 6.179444349519102e-05
